# Forecasting with Litterman

This tutorial shows how to produce probabilistic forecasts from a fitted Bayesian VAR.

In [ ]:
import numpy as np
import pandas as pd

from litterman import VAR, VARData
from litterman.samplers import NUTSSampler

## Simulate and fit

We'll create a simple VAR(1) and fit it.

In [ ]:
rng = np.random.default_rng(42)
T = 200
y = np.zeros((T, 2))
for t in range(1, T):
    y[t] = 0.5 * y[t - 1] + rng.standard_normal(2) * 0.1

index = pd.date_range("2000-01-01", periods=T, freq="QS")
data = VARData(endog=y, endog_names=["gdp", "inflation"], index=index)

sampler = NUTSSampler(draws=500, tune=500, chains=2, cores=1, random_seed=42)
fitted = VAR(lags=1, prior="minnesota").fit(data, sampler=sampler)

## Produce forecasts

Call `.forecast(steps=h)` to get an h-step-ahead forecast.

In [ ]:
fcast = fitted.forecast(steps=8)
fcast.median()

## Credible intervals

Use `.hdi()` to get highest density intervals.

In [ ]:
hdi = fcast.hdi(prob=0.89)
print("Lower bounds:")
print(hdi.lower)
print("\nUpper bounds:")
print(hdi.upper)

## Convert to DataFrame

Use `.to_dataframe()` for a tidy format suitable for further analysis.

In [ ]:
fcast.to_dataframe()